In [18]:
def partition_integer(input_val, upper_bound, unit):
    # Initialize result list
    result = []
    
    # Calculate the maximum number of units we can use without exceeding the input value
    while input_val > 0:
        # Determine the maximum allowable partition value
        max_value = (input_val // unit) * unit
        
        if max_value == 0:
            # If max_value is 0, that means input_val is less than unit
            # Add the remainder to the result
            result.append(input_val)
            break
        
        # Take the minimum between max_value and upper_bound
        part = min(max_value, upper_bound)
        
        # If the part is a multiple of the unit, add it to the result
        if part % unit == 0:
            result.append(part)
            input_val -= part
        else:
            # If not, reduce part until it is a multiple of the unit
            part = (part // unit) * unit
            if part > 0:
                result.append(part)
                input_val -= part
            else:
                # If we cannot partition further due to a small input value, break the loop
                result.append(input_val)
                break

    return result

# Example usage
print(partition_integer(20, 4, 4))  # Output: [4, 4, 4, 4, 4]
print(partition_integer(10, 5, 2))  # Output: [4, 4, 2]
print(partition_integer(6, 4, 2))   # Output: [4, 2]
print(partition_integer(3, 5, 2))   # Output: [3]
print(partition_integer(5, 2, 2))   # Output: [5]


[4, 4, 4, 4, 4]
[4, 4, 2]
[4, 2]
[2, 1]
[2, 2, 1]


In [1]:
import heapq

class Env:
    def __init__(self):
        # Initialize rooms and assets
        self.rooms = ["Kitchen", "Living Room", "Restroom", "Bedroom"]
        self.assets = {
            "Kitchen": ["Table", "Toaster", "Sink", "Refrigerator", "Stove", "Pan"],
            "Living Room": ["Table", "Sofa", "Television"],
            "Restroom": ["Laundry Basket", "Washing Machine", "Dryer", "Sink", "Toilet"],
            "Bedroom": ["Bed", "Wardrobe", "Desk"],
        }
        # Graph includes both rooms and assets
        self.graph = {room: {} for room in self.rooms}
        for room in self.assets:
            for asset in self.assets[room]:
                asset_node = f"{room}:{asset}"
                self.graph[asset_node] = {}
                # Add default cost for moving from room to asset
                self.add_transition(room, asset_node, 1)

    def __repr__(self):
        return str(self.graph)

    def add_transition(self, node1, node2, cost):
        if node1 in self.graph and node2 in self.graph:
            self.graph[node1][node2] = cost
            self.graph[node2][node1] = cost
        else:
            raise ValueError("Both nodes must be in the graph.")

    def gen_dummy(self):
        # Add room to room transitions
        self.add_transition("Kitchen", "Living Room", 1)
        self.add_transition("Living Room", "Restroom", 1)
        self.add_transition("Living Room", "Bedroom", 1)
        self.add_transition("Bedroom", "Restroom", 1)
        
        # Add asset to asset transitions within rooms
        self.add_transition("Kitchen:Table", "Kitchen:Toaster", 1)
        self.add_transition("Kitchen:Toaster", "Kitchen:Sink", 1)
        self.add_transition("Kitchen:Sink", "Kitchen:Refrigerator", 1)
        self.add_transition("Kitchen:Stove", "Kitchen:Pan", 1)
        self.add_transition("Kitchen:Sink", "Kitchen:Stove", 1)

        self.add_transition("Living Room:Table", "Living Room:Sofa", 1)
        self.add_transition("Living Room:Sofa", "Living Room:Television", 1)
        
        self.add_transition("Restroom:Laundry Basket", "Restroom:Washing Machine", 1)
        self.add_transition("Restroom:Washing Machine", "Restroom:Dryer", 1)
        self.add_transition("Restroom:Sink", "Restroom:Toilet", 1)

        self.add_transition("Bedroom:Bed", "Bedroom:Wardrobe", 1)
        self.add_transition("Bedroom:Wardrobe", "Bedroom:Desk", 1)

        # Add inter-room asset transitions
        self.add_transition("Kitchen:Table", "Living Room:Table", 2)
        self.add_transition("Living Room:Sofa", "Bedroom:Bed", 2)

    def get_cost(self, departure: str, destination: str) -> int:

        def dijkstra(start, goal):
            # Priority queue: (cost, node)
            queue = [(0, start)]
            visited = {}
            while queue:
                current_cost, current_node = heapq.heappop(queue)
                if current_node in visited:
                    continue
                visited[current_node] = current_cost
                if current_node == goal:
                    return current_cost

                for neighbor, cost in self.graph[current_node].items():
                    if neighbor not in visited:
                        heapq.heappush(queue, (current_cost + cost, neighbor))

            raise ValueError("No path found between the given nodes.")

        # Normalize input for asset queries
        if ":" not in departure:
            departure = self.normalize_room_or_asset(departure)
        if ":" not in destination:
            destination = self.normalize_room_or_asset(destination)

        return dijkstra(departure, destination)

    def normalize_room_or_asset(self, name: str) -> str:
        # Normalize names to include room:asset notation if needed
        for room, assets in self.assets.items():
            if name == room:
                return room
            if name in assets:
                return f"{room}:{name}"
        raise ValueError(f"Name {name} not found in rooms or assets.")

# Example usage:
env = Env()
env.gen_dummy()
print(env.get_cost("Kitchen:Table", "Kitchen:Sink"))  # Should output 1
print(env.get_cost("Kitchen", "Living Room:Table"))  # Should output 2


2
2


In [2]:
inputs = [(0,False)]
for time_slot, is_urgency in inputs:
    print(time_slot, is_urgency)

0 False


In [1]:
from typing import List

import networkx as nx
from anytree import Node

from concept.agent import Agent
from concept.task import Subtask, Task, get_all_subtasks
from task_management.handler.constraint_handler import ConstraintHandler
from task_management.handler.dynamic_task_handler import TaskHandler


class TreeBuilder:
    def __init__(
        self,
        agent: Agent,
        tasks: List[Task],
        task_handler: TaskHandler,
        constraints: nx.DiGraph,
    ):
        self.agent = agent
        self.tasks = tasks
        self.task_handler = task_handler
        self.constraint_handler = ConstraintHandler(agent, constraints)

    def build_tree(self) -> Node:
        root_node = Node(name="Start", makespan=0, location=self.agent.location)
        subtasks = get_all_subtasks(self.tasks)
        initial_subtasks = self._get_initial_subtasks(subtasks)

        for subtask in initial_subtasks:
            remaining_subtasks = subtasks[:]
            remaining_subtasks.remove(subtask)
            self._add_subtask_to_tree(root_node, subtask, remaining_subtasks)

        return root_node

    def _get_initial_subtasks(self, subtasks: List[Subtask]) -> List[Subtask]:
        initial_nodes = {
            node
            for node, in_degree in self.constraint_handler.constraints.in_degree()
            if in_degree == 0
        }
        return [subtask for subtask in subtasks if subtask.name in initial_nodes]

    def _add_subtask_to_tree(
        self, parent_node: Node, subtask: Subtask, remaining_subtasks: List[Subtask]
    ) -> None:

        # Retrieve the makespan and location from the parent node
        makespan = parent_node.makespan
        self.agent.location = parent_node.location

        # Handle movement
        parent_node, makespan = self.task_handler.handle_movement(
            parent_node, subtask, makespan
        )

        # Check for time slot handling needs
        time_slot_urgencies = self.constraint_handler.get_time_slot_and_urgency(
            parent_node, subtask
        )

        # Process each time slot urgency
        for time_slot, is_urgency in time_slot_urgencies:
            if time_slot > 0:
                if is_urgency:
                    # Check if other quick tasks can be performed in this time slot
                    available_subtasks = self._get_eligible_subtasks(
                        parent_node, remaining_subtasks
                    )
                    time_spent = 0

                    for available_subtask in available_subtasks:
                        if (
                            available_subtask.duration.interval
                            <= time_slot - time_spent
                        ):
                            self._add_subtask_to_tree(
                                parent_node, available_subtask, remaining_subtasks
                            )
                            time_spent += available_subtask.duration.interval
                            remaining_subtasks.remove(available_subtask)

                            if time_spent >= time_slot:
                                break

                    # Add waiting time if there is still some time left after quick tasks
                    if time_spent < time_slot:
                        wait_time = time_slot - time_spent
                        wait_node = Node(
                            name=f"Wait_for_{subtask.name}",
                            parent=parent_node,
                            makespan=makespan + wait_time,
                            location=parent_node.location,
                        )
                        makespan += wait_time
                        parent_node = wait_node
                else:
                    # If urgency is False, just wait the specified time
                    wait_node = Node(
                        name=f"Wait_for_{subtask.name}",
                        parent=parent_node,
                        makespan=makespan + time_slot,
                        location=parent_node.location,
                    )
                    makespan += time_slot
                    parent_node = wait_node

        # Add the subtask execution
        makespan += subtask.duration.interval
        child_node = Node(
            subtask.name,
            parent=parent_node,
            makespan=makespan,
            location=f"{subtask.roi.room}:{subtask.roi.asset}",
        )

        # Expand the tree with remaining subtasks
        self._expand_tree(child_node, remaining_subtasks)

    def _expand_tree(
        self, parent_node: Node, remaining_subtasks: List[Subtask]
    ) -> None:
        eligible_subtasks = self._get_eligible_subtasks(parent_node, remaining_subtasks)

        for subtask in eligible_subtasks:
            new_remaining_subtasks = remaining_subtasks[:]
            new_remaining_subtasks.remove(subtask)
            self._add_subtask_to_tree(parent_node, subtask, new_remaining_subtasks)

    def _get_eligible_subtasks(
        self, parent_node: Node, remaining_subtasks: List[Subtask]
    ) -> List[Subtask]:

        results = []

        for subtask in remaining_subtasks:
            if self.constraint_handler.validate_ordering_constraints(
                parent_node, subtask
            ):
                time_slot_urgencies = self.constraint_handler.get_time_slot_and_urgency(
                    parent_node, subtask
                )

                if self.constraint_handler.validate_timing_constraints(
                    time_slot_urgencies
                ):
                    results.append(subtask)

        return results
